# Single-Engine OpenVINO int8 Semantic Grounder

**Author:** kj  **Approach:** one inference engine (OpenVINO), int8 throughout, configurable top-k pre-filter

The deployable semantic grounder on a single runtime. A bi-encoder pre-filters the evidence
chunks to the top-k most relevant, then two cross-encoders - a relevance reranker and an NLI
entailment model - score the survivors, and a logistic over their max-over-chunks scores
classifies each claim as supported or hallucination. Every model runs as an **OpenVINO int8 IR**:
no PyTorch, no ONNX Runtime, no mixed runtime.

1. **Build / load** the three OpenVINO int8 IRs (embedder, reranker, NLI) - cached under `models/ov/`, push-ready
2. **Parity** - each int8 IR vs its cached fp32 scores
3. **Pipeline scoring** - all chunks scored once on an eval subset (with progress bars), cached
4. **Quality vs top-k** and **latency vs top-k** - the quality/speed tradeoff of the pre-filter

The full-gold stack macro-F1 (0.795, int8) is established in `02-kj-deberta-int8-smoothquant.ipynb`;
this notebook adds the single-engine pipeline and the top-k tradeoff, evaluated on a stratified
subset so the in-notebook chunk scoring stays watchable.

**Portability:** OpenVINO runs on x86-64 (Intel and AMD; int8 via AVX2 / AVX-512-VNNI) and on ARM
(aarch64 / Graviton via the ARM CPU plugin - less mature, validate on the target).

## Environment

CPU-only OpenVINO; models are already in the local HF cache.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""        # CPU-only (OpenVINO)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import time
from pathlib import Path

# Data processing
import numpy as np
from scipy.stats import pearsonr

# OpenVINO + tokenizers
import openvino as ov
# Stack meta-classifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

# Project
from grounding_models import load_gold, SCORES_DIR, ROOT
from grounding_ensemble import best_macro
from grounding_openvino import (load_ov_hf, HF_REPOS, embed_vectors,
                                                 topk_chunks, rerank_max, nli_max,
                                                 score_pipeline_ranked)

# Rich console output + progress
from rich import print as rprint
from tqdm.auto import tqdm

## Reproducibility

In [ ]:
SEED = 42
np.random.seed(SEED)

## Configuration

`TOP_K` is the deployed pre-filter width. Quality and latency are reported for every k in
`K_SWEEP` (50 = all chunks). Pipeline scoring runs on a stratified `EVAL_N` subset so the
in-notebook chunk calculation completes in a watchable time; the full-gold int8 macro-F1 (0.795)
is in notebook 02.

In [ ]:
# Models (all -> OpenVINO int8 IR under models/ov/<name>/)
EMBEDDER_ID, EMBEDDER_POOL = "BAAI/bge-m3", "cls"        # bi-encoder pre-filter
RERANKER_ID = "BAAI/bge-reranker-v2-m3"                  # cross-encoder relevance
NLI_ID, NLI_ALPHA, ENT_IDX = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", 0.7, 0

# Pipeline
TOP_K = 8                                                # deployed pre-filter width (configurable)
K_SWEEP = [5, 8, 12, 50]                                 # top-k values (50 == all chunks)
CHUNK_SIZE, CHUNK_OVERLAP, MAX_CHUNKS = 1100, 200, 50

# Eval
EVAL_N = 800                                             # stratified subset for pipeline scoring
N_PARITY = 48                                            # parity sample (balanced)
LAT_N = 20                                               # records for the latency micro-benchmark
N_CALIB = 128
TARGET_MACRO_F1 = 0.782                                  # one fold-std below fp32 stack 0.796

# Paths
MODELS = ROOT / "models" / "ov"; MODELS.mkdir(parents=True, exist_ok=True)
PIPE_CACHE = SCORES_DIR / f"ov_pipeline_sub{EVAL_N}.npz"
core = ov.Core()

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"-" * 44}[/dim]
[bold]Models (OpenVINO int8)[/bold]
  Embedder: [cyan]{EMBEDDER_ID}[/cyan] [dim](pre-filter)[/dim]
  Reranker: [cyan]{RERANKER_ID}[/cyan]
  NLI: [cyan]{NLI_ID}[/cyan] [dim](SmoothQuant alpha {NLI_ALPHA})[/dim]

[bold]Pipeline[/bold]
  Deployed top-k: [yellow]{TOP_K}[/yellow] [dim](sweep {K_SWEEP})[/dim]
  Eval subset: [yellow]{EVAL_N}[/yellow] records

[bold]Runtime[/bold]
  Device: [green]CPU[/green] [dim](OpenVINO {ov.__version__})[/dim]
""")

## Data Loading

Loads the gold and cached fp32 reference scores, then draws a balanced `EVAL_N` subset for pipeline scoring.

In [ ]:
recs_all = load_gold(CHUNK_SIZE, CHUNK_OVERLAP, MAX_CHUNKS)
y_all = np.array([r["label"] for r in recs_all])
cached_emb = np.load(SCORES_DIR / "BAAI__bge-m3.npy")
cached_rr = np.load(SCORES_DIR / "BAAI__bge-reranker-v2-m3.npy")
cached_nli = np.load(SCORES_DIR / "MoritzLaurer__mDeBERTa-v3-base-mnli-xnli.npy")

eval_idx = np.sort(np.concatenate([
    np.random.choice(np.where(y_all == 0)[0], EVAL_N // 2, replace=False),
    np.random.choice(np.where(y_all == 1)[0], EVAL_N // 2, replace=False)]))
recs = [recs_all[i] for i in eval_idx]; y = y_all[eval_idx]

half = N_PARITY // 2
parity_idx = np.sort(np.concatenate([
    np.random.choice(np.where(y_all == 0)[0], half, replace=False),
    np.random.choice(np.where(y_all == 1)[0], half, replace=False)]))
calib_idx = np.random.choice(len(recs_all), N_CALIB, replace=False)

rprint(f"""[white]Data[/white]
  Gold: [yellow]{len(recs_all)}[/yellow]   Eval subset: [yellow]{len(recs)}[/yellow] [dim]({int((y==0).sum())} hall / {int((y==1).sum())} supp)[/dim]
  Total chunks to score: [yellow]{sum(len(r['chunks']) for r in recs)}[/yellow]
""")

## Load OpenVINO int8 models from HuggingFace

Pulls the three published OpenVINO int8 IRs from the Hub (`stellars/*-openvino-int8`) - IR + config
+ tokenizer ship together, so no base model or local build is needed. Cached after first download.
They were quantized by `scripts/build_ov_grounder.py` and published with `scripts/push_ov_models_to_hf.py`.

In [ ]:
irs, toks = {}, {}
for name, repo in HF_REPOS.items():
    cm, tok, d = load_ov_hf(repo)                       # download + compile the int8 IR
    irs[name], toks[name] = cm, tok
    mb = sum(p.stat().st_size for p in d.glob("*.bin")) / 1e6
    rprint(f"  [cyan]{name}[/cyan] <- [dim]{repo}[/dim]: [green]{mb:.0f} MB[/green]")

## Parity check

Each int8 IR vs its cached fp32 scores on a small balanced sample. >= 0.95 means quantization kept the signal.

In [ ]:
def _emb_max(cm, tok, r):
    cv = embed_vectors(cm, tok, [r["claim"]], pool=EMBEDDER_POOL)[0]
    dv = embed_vectors(cm, tok, r["chunks"], pool=EMBEDDER_POOL)
    return float((dv @ cv).max())

rows = []
for name, fn, ref in [("bge-m3", _emb_max, cached_emb),
                      ("bge-reranker-v2-m3", lambda cm, tk, r: rerank_max(cm, tk, r["claim"], r["chunks"]), cached_rr),
                      ("mDeBERTa-v3-nli", lambda cm, tk, r: nli_max(cm, tk, r["claim"], r["chunks"]), cached_nli)]:
    sc = np.array([fn(irs[name], toks[name], recs_all[i]) for i in tqdm(parity_idx, desc=name)])
    rows.append((name, float(pearsonr(sc, ref[parity_idx])[0]), sc.std()))

body = "\n".join(f"  [cyan]{n}[/cyan]: pearson [{'green' if p>=0.95 else 'yellow'}]{p:.4f}[/]  std {s:.3f}"
                  for n, p, s in rows)
rprint(f"[bold cyan]int8 parity vs fp32 ({N_PARITY} sample)[/bold cyan]\n[dim]{'-'*44}[/dim]\n{body}\n")

## Pipeline scoring (chunk calculations, with progress bars)

Scores every chunk of the eval subset once - embed (claims + chunks), reranker, NLI - via OpenVINO
async queues, each stage with a progress bar. Then takes the max reranker / NLI over the top-k per
record for every k, so the top-k sweep below is a free slice. Cached, so reruns load instantly.

In [ ]:
if PIPE_CACHE.exists():
    d = np.load(PIPE_CACHE); rerank_k, nli_k, KS = d["rerank_k"], d["nli_k"], list(d["ks"])
    rprint(f"[dim]loaded cached pipeline scores: {PIPE_CACHE.name}[/dim]")
else:
    t = time.time()
    rerank_k, nli_k = score_pipeline_ranked(
        recs, irs["bge-m3"], toks["bge-m3"],
        irs["bge-reranker-v2-m3"], toks["bge-reranker-v2-m3"],
        irs["mDeBERTa-v3-nli"], toks["mDeBERTa-v3-nli"],
        K_SWEEP, ent_idx=ENT_IDX, pool=EMBEDDER_POOL, progress=True)
    KS = K_SWEEP
    np.savez(PIPE_CACHE, rerank_k=rerank_k, nli_k=nli_k, ks=np.array(KS))
    rprint(f"[green]scored {len(recs)} records[/green] in [yellow]{(time.time()-t)/60:.1f} min[/yellow] -> cached {PIPE_CACHE.name}")

## Quality vs top-k

Stack macro-F1 (out-of-fold on the eval subset) at each top-k. Flat across k means the pre-filter
can prune aggressively with no quality loss.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
def stack_macro(rr, nli):
    X = np.column_stack([rr, nli])
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    p = cross_val_predict(clf, X, y, cv=skf, method="predict_proba")[:, 1]
    return best_macro(p, y)[0], roc_auc_score(y, p)

qual = [(k, *stack_macro(rerank_k[:, j], nli_k[:, j])) for j, k in enumerate(KS)]
jdep = KS.index(TOP_K)
mf1, auc = qual[jdep][1], qual[jdep][2]; ok = mf1 >= TARGET_MACRO_F1
body = "\n".join(f"  top-k [yellow]{k:>2}[/yellow]: macro-F1 [cyan]{mf:.3f}[/cyan]  AUC {au:.3f}"
                  + ("  [green]<- deployed[/green]" if k == TOP_K else "") for k, mf, au in qual)
rprint(f"""[bold cyan]Stack macro-F1 vs top-k (eval subset, OOF)[/bold cyan]
[dim]{"-"*48}[/dim]
{body}

  Deployed k={TOP_K}: [{'green' if ok else 'red'}]{mf1:.3f}[/] (AUC {auc:.3f})  gate >= {TARGET_MACRO_F1}: [{'green' if ok else 'red'}]{'PASS' if ok else 'FAIL'}[/]
  [dim]full-gold int8 stack (notebook 02): macro-F1 0.795[/dim]
""")

## Latency - ms/claim vs top-k

Times the live single-engine pipeline (embed pre-filter then the two cross-encoders) on a small
sample at each top-k. Lower k = fewer cross-encoder passes = faster.

In [ ]:
lat_idx = np.random.choice(len(recs), LAT_N, replace=False)
lat_recs = [recs[i] for i in lat_idx]

def time_pipeline(records, k):
    t = time.time()
    for r in records:
        ch = topk_chunks(irs["bge-m3"], toks["bge-m3"], r["claim"], r["chunks"], k, pool=EMBEDDER_POOL)
        rerank_max(irs["bge-reranker-v2-m3"], toks["bge-reranker-v2-m3"], r["claim"], ch)
        nli_max(irs["mDeBERTa-v3-nli"], toks["mDeBERTa-v3-nli"], r["claim"], ch)
    return (time.time() - t) / len(records) * 1000

_ = time_pipeline(lat_recs[:3], TOP_K)                  # warm up
lat = [(k, time_pipeline(lat_recs, k)) for k in tqdm(KS, desc="latency")]
body = "\n".join(f"  top-k [yellow]{k:>2}[/yellow]: [cyan]{ms:6.0f}[/cyan] ms/claim" for k, ms in lat)
slow = next(ms for k, ms in lat if k == max(KS)); fast = next(ms for k, ms in lat if k == TOP_K)
rprint(f"""[bold cyan]Latency vs top-k ({LAT_N} records, CPU)[/bold cyan]
[dim]{"-"*44}[/dim]
{body}

  top-k={TOP_K} vs all-chunks: [green]{slow/fast:.1f}x faster[/green] [dim]({fast:.0f} vs {slow:.0f} ms/claim)[/dim]
""")

## Summary

In [ ]:
best_q = max(qual, key=lambda q: q[1])
rprint(f"""[bold]Single-engine OpenVINO int8 grounder[/bold]
[dim]{"-"*52}[/dim]
  Engine: [green]OpenVINO[/green] [dim](one runtime: embedder + reranker + NLI all int8)[/dim]
  Pipeline: [cyan]bge-m3[/cyan] pre-filter -> [cyan]bge-reranker-v2-m3[/cyan] + [cyan]mDeBERTa-v3 NLI[/cyan] -> logistic
  Deployed top-k=[yellow]{TOP_K}[/yellow]: macro-F1 [green]{mf1:.3f}[/green] (AUC {auc:.3f}), [green]{fast:.0f}[/green] ms/claim [dim](eval subset {EVAL_N})[/dim]
  Quality flat across k [dim](best k={best_q[0]} @ {best_q[1]:.3f})[/dim]; pruning to k={TOP_K} is [green]{slow/fast:.1f}x[/green] faster than all-chunks
  Full-gold int8 macro-F1 (notebook 02): [green]0.795[/green]
  Models from HF [cyan]stellars/*-openvino-int8[/cyan]

  [dim]Portability: x86-64 Intel/AMD native; ARM via the OpenVINO ARM plugin (validate on target).[/dim]
""")